# Filtering Metadata

In [ ]:
import pandas as pd
import numpy as np
import openpyxl

# Load metadata from the specified input directory
metadata = pd.read_excel("/path/to/input/metadata_txt.xlsx", sheet_name="Sheet1", engine='openpyxl') 
# display(metadata)

# Exclusion Criteria: Remove subjects with Inflammatory Bowel Disease (IBD)
metadata = metadata[metadata['IBD'] == 0]

angina = metadata[metadata['Diagnosis'] == 'Angina']
mi = metadata[metadata['Diagnosis'] == 'MI']
hf = metadata[metadata['Diagnosis'] == 'HF']

# Unify ACS to MI for group comparison
metadata['Group'] = metadata['Group'].replace('ACS', 'MI')
print(f"Total (n={sum([len(angina), len(mi), len(hf)])})\nAngina (n={len(angina)})\nMI (n={len(mi)})\nHF (n={len(hf)})")

display(metadata)

In [ ]:
from sklearn.impute import KNNImputer

metadata_filt = metadata.copy()
final_sampleID = list(metadata['SampleID'])

imputer = KNNImputer(n_neighbors=3)

metadata_filt['eGFR_CKD_EPI'] = imputer.fit_transform(metadata_filt[['eGFR_CKD_EPI']])
metadata_filt['Tg'] = imputer.fit_transform(metadata_filt[['Tg']])
metadata_filt['Tchol'] = imputer.fit_transform(metadata_filt[['Tchol']])
metadata_filt['HDL'] = imputer.fit_transform(metadata_filt[['HDL']])
metadata_filt['LDL'] = imputer.fit_transform(metadata_filt[['LDL']])

display(metadata_filt)

# Filtering FeatureTable

In [ ]:
import pandas as pd

# Load raw metabolomics data
df = pd.read_excel("/path/to/input/Data.xlsx", sheet_name="Sheet1", engine='openpyxl')
display(df)
cols_met = list(df.columns[1:])
df['Group'] = metadata['Group']
df = df[['SampleID', 'Group'] + cols_met]
df = df[df['SampleID'].isin(final_sampleID)] # Exclude samples based on filtered metadata
df_for_ratio = df.copy()

# Filter features based on Quality Control (QC) Coefficient of Variation (CV)
cv_threshold = 30

qc = pd.read_excel("/path/to/input/Data_QC.xlsx", sheet_name="Sheet1", engine='openpyxl')
qc = qc[cols_met].T
qc_mean = qc.mean(axis=1)
qc_std = qc.std(axis=1)
qc_cv = qc_std / qc_mean * 100
qc['CV(%)'] = qc_cv
qc = qc[['CV(%)']]

qc = qc[qc['CV(%)'] < cv_threshold].T
cols_qc = list(qc.columns)
display(df)
df = df[['SampleID', 'Group'] + cols_qc]

# Export intermediate diversity data
df.to_csv('/path/to/output/data_diversity.csv', sep=',', index=False)
display(df)

# Load Limit of Detection (LOD) data and select LOD cutoff
lod_raw = pd.read_excel("/path/to/input/data_lod.xlsx", sheet_name="Sheet1", engine='openpyxl')
lod = pd.DataFrame(columns=cols_met)
for col in cols_met:
    pair = list(lod_raw[col])
    if pair[0] == 0:
        lod.loc['LOD', col] = pair[1]
    else: lod.loc['LOD', col] = pair[0]

# Split data by each group and apply the 80% rule (remove features missing in >20% of samples per group)
df['Group'] = df['Group'].replace('ACS', 'MI')
groups = ['Angina', 'MI', 'HF']
group_dict = {}

cols_exclude = list(df.columns[2:])

for gr in groups:
    group_dict[gr] = df[df['Group'] == gr]
    total_count = len(group_dict[gr])

    for col in group_dict[gr].columns[2:]:
        lod_by_col = lod.loc['LOD', col]
        vals_by_col = list(group_dict[gr][col])
        lower_than_lod_by_col = len([x for x in vals_by_col if x < lod_by_col])
        perc_lod = lower_than_lod_by_col / total_count
        if perc_lod <= 0.2:
            if col in cols_exclude:
                cols_exclude.remove(col)

df_final = df.copy()
df_final = df_final.drop(columns=cols_exclude)
display(df_final)

In [ ]:
row_sums = df_for_ratio.iloc[:,2:].sum(axis=1)
df_for_ratio.iloc[:,2:] = df_for_ratio.iloc[:,2:].div(row_sums, axis=0)
df_for_ratio = df_for_ratio[df_final.columns]
df_for_ratio = df_for_ratio[df_for_ratio['SampleID'].isin(df_final['SampleID'])]
display(df_for_ratio)

In [ ]:
# Export finalized metadata and feature tables
metadata.to_csv('/path/to/output/metadata.csv', sep=',', index=False)
metadata_filt.to_csv('/path/to/output/metadata_filt.csv', sep=',', index=False)
df_final.to_csv('/path/to/output/data.csv', sep=',', index=False)
df_for_ratio.to_csv('/path/to/output/data_rel_abun.tsv', sep='\t', index=False)